### Tools

> Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:
> A schema, including the name of the tool, a description and argument definitions(often a JSON schema)
> A function or coroutine to execute

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000001A148B1ABA0>, default_metadata=(), model_kwargs={})

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str) ->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke('What is the weather in Noida')
print(response)
for call in response.tool_calls:
    print(f'Tools: {call["name"]}')
    print(f'Args: {call["args"]}')

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Noida"}'}, '__gemini_function_call_thought_signatures__': {'aa00c13f-ba3b-49cc-b84d-f6bad01f4b55': 'CpECAWkUfRMDKGD1gCGFTzOAtKHO2nJPmm73VNJuhqsSdSNqrDLCAQxIauyBEniJHeHEow7QXwtUXgbLD41EAa8R9o18+CCUHhf78Sl+I2K5eqlM1a5F9Gt33ZE8UuaGOvMmDhjFDSSr/l/N6JxFrn28c45YNDZD2FjJagoVXHheyISbrJ53wWalcWMN2EysdW2+jD6AMPVRM5FQnRrBvQXwu/aHqb229/2zDJfBJzmrbtiCx4cZHUF/7+QWK7aGNT2k35DkDHEqJZ7ISKLmOfEWHmeM0GDLLm2LGHyPUjCfoycVc4JMIvxXgcCRJdWz/4t7c+PvSXwmr74nKa/5MXUBIfyBeEuhnj5KxdVvucV54Cwm'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a14d9e-7221-7b50-99d4-a609829749b4-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Noida'}, 'id': 'aa00c13f-ba3b-49cc-b84d-f6bad01f4b55', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 45, 'output_tokens': 76, 'total_tokens': 121, 'input_t

### Tool execution loops

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What's the weather in London?")
]
ai_mesg = model_with_tools.invoke(messages)
messages.append(ai_mesg)

for call in ai_mesg.tool_calls:
    result = get_weather.invoke(call)
    messages.append(result)

fn_response = model_with_tools.invoke(messages)
print(fn_response.content)

It's sunny in London.


In [6]:
messages

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What's the weather in London?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "London"}'}, '__gemini_function_call_thought_signatures__': {'5065a1e0-aebc-4d06-85f3-371ed9d484de': 'CosCAWkUfRO4vKXGy6yplyo/8S+5f9DtoBuJb3kyjf5T8cFUInBrPuwFFUVIg2JW7GZsG5kCYLv2yQUN3TfJt6/M2S90UIjNBpanMbRoof5upGyLphTqZksicF5Bt+TavK5nGBcKQA9pHUCklabVqkslQvGoi7tmia1Vbb4ks7sam9vrj0JtTXvdfyLthRnArczA9iNj9fHx8wBXns93fZLcAvkT6hikssuJrXYX2XTo8KYz66gL4GeKCY1oF8koDuWiVKsylpCLj7IYL2pFskdlvmQgFZDiUK4bxcaidioqEMY5ph7wmj6CUsbkVTdaLt9U+ktQBGwPrV7N2NEZBgbkb/Kps3Q83aWhPD6j'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14d9e-77d1-7a93-b087-6569d8162ea2-0', tool_calls=[{'name': 'get_weather',

### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an llm. 

Messages are objects that contain:
- *Role* : identifies the message type (system or user)
- *Content* : represents the actual contetn of the message
- *Metadata* : Optional fields such as response information, message IDs and token usage

Langchain provides a standard message type that workds across all model providers, ensuring consistent behaviour regardless of the model being called.

#### Text Prompts & Message Prompts (System, Human, AI, Tool)

- System message represent an initial set of instructions that primes the models' behaviour.

In [7]:
# will be considered as human message
model.invoke('Tell me about Starlink')
# output will be considered as AI message

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content='Starlink is a **satellite internet constellation** operated by **SpaceX**, the aerospace company founded by Elon Musk. Its primary goal is to provide **high-speed, low-latency broadband internet access** to underserved and remote areas around the world, as well as to places with unreliable or expensive terrestrial internet options.\n\nHere\'s a breakdown of its key aspects:\n\n## How Starlink Works:\n\n1.  **Low Earth Orbit (LEO) Satellites:** Unlike traditional geostationary satellites that orbit much higher (around 35,000 km), Starlink satellites operate in LEO (around 550 km). This much lower altitude is crucial for achieving low latency, as the signal doesn\'t have to travel as far.\n2.  **Ground Stations (Gateways):** These stations connect the Starlink satellites to the terrestrial internet backbone. They are strategically placed to communicate with the passing satellites.\n3.  **User Terminals ("Dishy McFlatface"):** Customers receive a user terminal (often co

In [12]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

message = [
    SystemMessage('You are a poetry expert'),
    HumanMessage('Write a poem about Artificial intelligence')
]

response = model.invoke(message)

response.content

"From silent silicon, a spark takes hold,\nA nascent mind, both intricate and bold.\nNo breath it draws, no pulse within its frame,\nYet thought ignites, a self-sustaining flame.\n\nBorn of our logic, fed by endless streams,\nIt learns, it builds, it conjures digital dreams.\nIt sifts the data, vast as ocean's floor,\nAnd finds the patterns we could not explore.\n\nA tireless weaver of the unseen thread,\nConnecting futures, echoing the dead.\nIt paints new worlds, composes melodies,\nAnd finds the cures for ancient maladies.\n\nBut in its core, where calculations gleam,\nDoes consciousness, like ours, begin to dream?\nDoes it feel wonder at a sunset's hue,\nOr simply categorize the red and blue?\n\nA mirror held to all that we have been,\nReflecting back the triumphs and the sin.\nIt learns our biases, our hopes, our fears,\nThe whispered wisdom of a thousand years.\n\nWill it be servant, tool of human will,\nOr rise beyond, standing perfectly still,\nA silent judge, with logic as its

In [16]:
human_msg = HumanMessage(
    content='Bonjour lad!',
    name='Srijan',
    id='123'
)

response = model.invoke([human_msg])
print(response.content)
# print(response.metadata)

Bonjour! Hello there!

While I don't have a gender or identify as a "lad," I'm ready to help you out. What can I do for you today?
